<b><font size="6" color="#E8800A">Week 5 · Model Selection</font></b><br>

A validation score guides model choices, so it cannot also provide an independent
assessment of the selected model. This notebook separates those jobs using a
train, validation and test split, then repeats the training and validation
comparison through cross-validation. Every comparison inherits the preprocessing
recipe recorded in Weeks 3 and 4.

<div class="alert alert-block alert-info">

## Table of Contents<a class="anchor" id="toc"></a>
### [<font color='#E8800A'>1 - Data Setup</font>](#setup)
* [<font color='#E8800A'>1.1. - Loading Libraries</font>](#libraries)
* [<font color='#E8800A'>1.2. - Loading the Dataset and Recipe</font>](#data-loading)
### [<font color='#E8800A'>2 - Holdout: Training, Validation and Test</font>](#train-test)
* [<font color='#E8800A'>2.1. - Reserving the Test Set</font>](#train-test-split)
* [<font color='#E8800A'>2.2. - Creating Training and Validation Sets</font>](#three-way-split)
### [<font color='#E8800A'>3 - Cross-Validation Techniques</font>](#cross-validation)
* [<font color='#E8800A'>3.1. - K-Fold Cross-Validation</font>](#kfold)
* [<font color='#E8800A'>3.2. - Repeated K-Fold Cross-Validation</font>](#repeated-kfold)
* [<font color='#E8800A'>3.3. - Leave-One-Out Cross-Validation</font>](#loo)
* [<font color='#E8800A'>3.4. - Group-Aware and Time-Aware Splitting</font>](#stratified)
* [<font color='#E8800A'>3.5. - Which Design Estimated Best?</font>](#frame-comparison)
### [<font color='#E8800A'>4 - Comparing Models</font>](#model-comparison)
* [<font color='#E8800A'>4.1. - Decision Tree Classifier</font>](#decision-tree)
* [<font color='#E8800A'>4.2. - Model Comparison Results</font>](#comparison-results)
### [<font color='#E8800A'>5 - Hyperparameter Tuning</font>](#hyperparameter-tuning)
* [<font color='#E8800A'>5.1. - What are Hyperparameters?</font>](#what-are-hyperparameters)
* [<font color='#E8800A'>5.2. - Hyperparameter Search with Holdout Validation</font>](#grid-search)
* [<font color='#E8800A'>5.3. - Grid Search with Cross-Validation</font>](#grid-search-cv)
* [<font color='#E8800A'>5.4. - Using Pipeline with GridSearchCV</font>](#pipeline)
* [<font color='#E8800A'>5.5. - The Two Execution Schemas</font>](#execution-schemas)
### [<font color='#E8800A'>Gold Standard: Nested Cross-Validation</font>](#nested-cv)
### [<font color='#E8800A'>The Complete Model Selection Workflow</font>](#workflow)
* [<font color='#E8800A'>Key Takeaways</font>](#takeaways)
### Optional (Advanced)
* [<font color='#E8800A'>Scikit-learn Hyperparameter Optimization Tools</font>](#optional-tools)
</div>

<a class="anchor" id="setup">

## <font color='#E8800A'>1. Data Setup</font>
</a>

<a class="anchor" id="libraries">

### <font color='#E8800A'>1.1. Loading Libraries</font>
</a>

__`Step 1`__ Import the libraries used for splitting, fitting and comparison.

[Back to TOC](#toc)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

from time import perf_counter

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    KFold,
    LeaveOneOut,
    RepeatedKFold,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline

from course_helpers import PLOT_BLUE, PLOT_ORANGE

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 33
np.random.seed(RANDOM_STATE)  # For reproducibility

import sys
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from preprocessing import (
    CachedSearchCV,
    PreparedEstimator,
    classification_preprocessor,
    load_classification,
)
CLASSIFICATION_RECIPE_LOG = "../../logs/week_04_feature_work_classification_log.json"


<a class="anchor" id="data-loading">

### <font color='#E8800A'>1.2. Loading the Dataset and Recipe</font>
</a>

`champions.csv` holds the rows kept in Week 3, missing values still open.
`load_classification` loads it with the dtypes recorded in the Week 4 log, and
`preprocessing.py` rebuilds the log's supported rules for fitting inside each
new split.

__`Step 2`__ Load `champions.csv` and its Week 4 recipe.

Run this notebook from its own folder inside the course
repository: that is what makes the `data/` paths below work. If you
downloaded this file on its own from Moodle, move it into the repository before
you run it.

In [ ]:
champions, classification_recipe = load_classification(
    '../../data/interim/champions.csv',
    CLASSIFICATION_RECIPE_LOG,
)
# The loader restores the nullable dtypes recorded in the log.
print(f'{champions.shape[0]:,} rows x {champions.shape[1]} columns')
champions.head()

__`Step 3`__ Separate the target `Outcome` from the inherited feature columns.

`RecordID` and `Athlete Id` identify rows, so the recipe's categorical and
numeric lists leave them out.

In [ ]:
# Outcome is the classification target. Keep independent copies.
target = champions['Outcome'].copy()
categorical_cols = list(classification_recipe['categorical'])
numerical_cols = list(classification_recipe['numeric'])
feature_columns = [column for column in champions.columns
                   if column in categorical_cols + numerical_cols]
data = champions.loc[:, feature_columns].copy()

print('Target:', target.name)
print(f'{len(categorical_cols)} categorical + {len(numerical_cols)} numeric features')

In [ ]:
# Read the inherited settings; this week changes the evaluation design.
print('Fill rules:', classification_recipe['fill'])
print('log1p columns:', classification_recipe['log1p'])
print('Encoding:', classification_recipe['encoding'])
print('Scaling:', classification_recipe['scaler'])
print('Selection:', classification_recipe['strategy']['kind'])

<a class="anchor" id="train-test">

## <font color='#E8800A'>2. Holdout: Training, Validation and Test</font>
</a>

Training rows fit the recipe and the model, validation scores guide the model
and hyperparameter choices, and the test set assesses the result once selection
is complete. Splitting before any preprocessing keeps both held-out sets out of
every fit.

<a class="anchor" id="train-test-split">

### <font color='#E8800A'>2.1. Reserving the Test Set</font>
</a>

Stratifying by `Outcome` gives both parts the same share of wins.

[Back to TOC](#toc)


<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html'>sklearn.model_selection.train_test_split(*arrays, test_size=None, train_size=None, random_state=None, shuffle=True, stratify=None)</a>

**Definition:**  
Split arrays or matrices into random train and test subsets.

**Common Parameters:**  
- `test_size`: Share of the rows that goes to the test split
- `random_state`: Seed that makes the shuffle reproducible
- `shuffle`: Whether to shuffle the rows before splitting
- `stratify`: Class labels whose proportions both subsets keep
</div>

__`Step 4`__ Reserve 20% for testing; keep the other 80% in `X_train_val` and `y_train_val`.

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(data, 
                                                    target, 
                                                    test_size=0.2, 
                                                    random_state=RANDOM_STATE, 
                                                    shuffle=True, 
                                                    stratify=target
                                                   )

A project competition may supply this test partition separately; here it comes from the labelled data.

<a class="anchor" id="three-way-split">

### <font color='#E8800A'>2.2. Creating Training and Validation Sets</font>
</a>

Taking one quarter of `X_train_val` for validation makes the validation set as
large as the test set.

__`Step 5`__ Split `X_train_val` and `y_train_val`, stratifying by `y_train_val`.

In [ ]:
# DO IT
X_train, X_val, y_train, y_val = train_test_split(X_train_val,
                                                  y_train_val,
                                                  test_size = 0.25,
                                                  random_state = RANDOM_STATE,
                                                  shuffle=True,
                                                  stratify=y_train_val
)

__`Step 6`__ Check the proportion of data for each dataset. _(written for you)_

In [ ]:
print(f'train: {len(y_train) / len(target):.0%} | '
      f'validation: {len(y_val) / len(target):.0%} | '
      f'test: {len(y_test) / len(target):.0%}')

__`Step 7`__ Fit the inherited recipe on training rows and transform all three sets.

`classification_preprocessor` turns the logged recipe into one preprocessing
object. `fit_transform(X_train, y_train)` learns the fitted values and selected
columns from training rows alone; `transform` applies that same fit to
validation and test rows.

In [ ]:
holdout_preprocessor = classification_preprocessor(classification_recipe)
X_train_prepared = holdout_preprocessor.fit_transform(X_train, y_train)
X_val_prepared = holdout_preprocessor.transform(X_val)
X_test_prepared = holdout_preprocessor.transform(X_test)

print('encoded columns:', len(holdout_preprocessor.encoded_names_))
print('kept by the selection:', X_train_prepared.shape[1])

__`Step 8`__ Fit a classifier on the training rows and score it on the validation and test rows.

The validation score is the estimate, and every choice below may read it. The
test score is what that estimate tries to predict, and no choice below may read
it.

Every score in this notebook is the F1 of the positive class, a win
(`Outcome` = 1). Predicting a win for every test row, printed last, sets the
floor a classifier has to clear.

In [ ]:
holdout_model = LogisticRegression(max_iter=1000)
holdout_model.fit(X_train_prepared, y_train)
holdout_val_predictions = holdout_model.predict(X_val_prepared)
holdout_test_predictions = holdout_model.predict(X_test_prepared)

holdout_score = f1_score(y_val, holdout_val_predictions)
holdout_test_score = f1_score(y_test, holdout_test_predictions)

# Section 3 changes the validation design and keeps the model, so every design
# from here on records what it estimated beside what it was estimating, and
# reprints the whole record.
frame_log = [{'frame': 'holdout', 'validation F1 is': 'untuned',
              'rows': len(y_train), 'folds': 1,
              'train F1': float(f1_score(y_train, holdout_model.predict(X_train_prepared))),
              'validation F1': float(holdout_score),
              'test F1': float(holdout_test_score),
              'model': repr(holdout_model)}]


def show_log(log):
    """Print every row of a log, and how far each estimate landed from its test F1."""
    table = pd.DataFrame(log)
    table['estimate minus truth'] = table['validation F1'] - table['test F1']
    table['absolute error'] = table['estimate minus truth'].abs()
    # The model's name is the longest column, so it goes last, left-aligned.
    table = table[[column for column in table if column != 'model'] + ['model']]
    width = table['model'].str.len().max()
    print(table.to_string(
        index=False,
        formatters={'train F1': '{:.4f}'.format,
                    'validation F1': '{:.4f}'.format,
                    'test F1': '{:.4f}'.format,
                    'estimate minus truth': '{:+.4f}'.format,
                    'absolute error': '{:.4f}'.format,
                    'model': lambda name: name.ljust(width)}))


print(f'one holdout split, {len(y_train)} train / {len(y_val)} validation'
      f' / {len(y_test)} test rows\n')
show_log(frame_log)

always_win = np.ones(len(y_test), dtype=int)
print(f'\nshare of wins in the test rows: {y_test.mean():.0%}')
print(f'F1 of predicting a win for every test row: {f1_score(y_test, always_win):.4f}')

<a class="anchor" id="cross-validation">

## <font color='#E8800A'>3. Cross-Validation Techniques</font>
</a>

One holdout score depends on which rows the split happened to draw, so
cross-validation averages over several splits of `X_train_val`. Each fold refits
the inherited recipe and a logistic regression on its own training rows; the
test set stays out.

<a class="anchor" id="kfold">

### <font color='#E8800A'>3.1. K-Fold Cross-Validation</font>
</a>

K-Fold divides the rows into K folds, and each fold validates once while the
other K-1 supply the training rows. The folds share training rows, so the K
scores are not independent.

<img src="https://scikit-learn.org/stable/_images/grid_search_cross_validation.png" alt="Training and validation folds" style="width: 500px;"/>

[Back to TOC](#toc)


The classifier and the inherited recipe stay fixed across splitters, so only the validation design changes.

`KFold` splits the rows into K consecutive folds; each fold validates once while the other K-1 train.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html'>sklearn.model_selection.KFold(n_splits=5, shuffle=False, random_state=None)</a>

**Definition:**  
K-Folds cross-validator. Provides train/test indices to split data into train/test sets.

**Common Parameters:**  
- `n_splits`: Number of folds (must be at least 2)
- `shuffle`: Whether to shuffle the rows before splitting; without it the folds are consecutive
</div>

`StratifiedKFold` gives every fold the class proportions of the whole dataset, which matters most when one class is rare.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html'>sklearn.model_selection.StratifiedKFold(n_splits=5, shuffle=False, random_state=None)</a>

**Definition:**  
Stratified K-Folds cross-validator. Provides train/test indices to split data into train/test sets while preserving the percentage of samples for each class.

**Common Parameters:**  
- `n_splits`: Number of folds (must be at least 2)
</div>

__`Step 9`__ Do steps 7 and 8 again as one object, and check the two agree.

In [ ]:
# Steps 7 and 8 fitted the recipe, transformed three matrices, then
# fitted a classifier on the first of them. PreparedEstimator is a custom
# class in preprocessing.py that does exactly that.
prepared = PreparedEstimator(
    classification_preprocessor(classification_recipe),
    LogisticRegression(max_iter=1000),
).fit(X_train, y_train)

print(f'by hand, steps 7-8 : {holdout_score:.4f}')
print(f'PreparedEstimator  : {f1_score(y_val, prepared.predict(X_val)):.4f}')

__`Step 10`__ Define two helpers. `avg_score(method, X, y, model=None, groups=None, pooled=False)` scores F1 on both sides of every fold and returns the winning model refitted on every row. `record(log, result, X_test, y_test)` scores that model once on the test rows and adds its row to a log.

In [ ]:
def avg_score(method, X, y, model=None, groups=None, pooled=False):
    """Cross-validate one model, or choose among several, and return it fitted.

    `model` is an unfitted estimator, an untuned logistic regression by
    default, or a list of candidates. Every fold refits the recipe and the
    model on that fold's training rows alone and scores both sides by F1.
    With a list, the first candidate with the highest mean validation F1 wins.

    `groups` reaches the splitters that need it, such as GroupKFold. With
    `pooled`, validation F1 is computed once over every held-out prediction,
    which folds of a single row need.

    Returns a dict: the winner's 'name', its per-fold 'train' and
    'validation' F1, the 'frame', 'rows' and 'folds' it was measured on, and
    the winner refitted on every row of X as 'model', ready to predict.
    """
    if isinstance(model, list):
        candidates = model
    else:
        candidates = [LogisticRegression(max_iter=1000) if model is None else model]
    # One set of folds for every candidate, so they compete on the same rows.
    folds = list(method.split(X, y, groups))

    best = None
    for candidate in candidates:
        score_train, held_out = [], []
        for train_index, val_index in folds:
            X_train, X_val = X.iloc[train_index], X.iloc[val_index]
            y_train, y_val = y.iloc[train_index], y.iloc[val_index]

            fitted = PreparedEstimator(
                classification_preprocessor(classification_recipe),
                clone(candidate),
            ).fit(X_train, y_train)
            score_train.append(f1_score(y_train, fitted.predict(X_train)))
            held_out.append((y_val, fitted.predict(X_val)))

        if pooled:
            held_out = [(pd.concat([truth for truth, _ in held_out]),
                         np.concatenate([guess for _, guess in held_out]))]
        score_val = [f1_score(truth, guess) for truth, guess in held_out]
        # The other metrics are reported beside F1; only F1 chooses.
        other = {label: np.mean([metric(truth, guess) for truth, guess in held_out])
                 for label, metric in (('accuracy', accuracy_score),
                                       ('precision', precision_score),
                                       ('recall', recall_score))}

        name = repr(candidate)
        spread = ' (pooled)' if pooled else f' +/- {np.std(score_val):.4f}'
        print(f'train F1 {np.mean(score_train):.4f} | validation accuracy '
              f"{other['accuracy']:.4f}, precision {other['precision']:.4f}, "
              f"recall {other['recall']:.4f}, F1 {np.mean(score_val):.4f}{spread} | {name}")
        # Strictly greater, so on a tie the earlier candidate keeps the lead.
        if best is None or np.mean(score_val) > np.mean(best['validation']):
            best = {'name': name, 'candidate': candidate,
                    'train': score_train, 'validation': score_val}

    best['model'] = PreparedEstimator(
        classification_preprocessor(classification_recipe),
        clone(best.pop('candidate')),
    ).fit(X, y)
    best.update(frame=type(method).__name__, rows=len(y), folds=len(folds))
    return best


def record(log, result, X_test, y_test, validation_is='untuned'):
    """Score a returned model once on the test rows, add its row and print the log."""
    row = {
        'frame': result['frame'],
        'validation F1 is': validation_is,
        'rows': result['rows'],
        'folds': result['folds'],
        'train F1': float(np.mean(result['train'])),
        'validation F1': float(np.mean(result['validation'])),
        'test F1': float(f1_score(y_test, result['model'].predict(X_test))),
        'model': result['name'],
    }
    # Running a cell again replaces its row rather than adding a second one.
    key = (row['frame'], row['validation F1 is'], row['model'])
    log[:] = [old for old in log
              if (old['frame'], old['validation F1 is'], old['model']) != key]
    log.append(row)
    show_log(log)

Each fold builds a fresh `PreparedEstimator` from the logged recipe, so only that fold's training rows fit the fills, categories, scale, selected columns and classifier; the validation rows are only scored.

__`Step 11`__ Create `kf`, a `KFold` with `n_splits=10`.

In [ ]:
kf = KFold(n_splits=10)

__`Step 12`__ Create `skf`, a `StratifiedKFold` with `n_splits=10`.

In [ ]:
skf = StratifiedKFold(n_splits=10)

__`Step 13`__ Score `kf` with __avg_score__, `record` the result in `frame_log`, and show the per-fold validation F1.

In [ ]:
kf_result = avg_score(kf, X_train_val, y_train_val)
record(frame_log, kf_result, X_test, y_test)
kf_result['validation']

__`Step 14`__ Score `skf` with __avg_score__ and `record` the result in `frame_log`.

In [ ]:
skf_result = avg_score(skf, X_train_val, y_train_val)
record(frame_log, skf_result, X_test, y_test)

**The single split was reading its own luck.** The Section 2 holdout scored
**0.8379**; ten folds of the same 3,200 rows average **0.8527**, and
stratifying them gives **0.8546**. The model settings did not change, only
which rows trained and validated it.

The fold scores run from **0.8269** to **0.8750**, so a single draw from that
range is a poor estimate of its middle. **Averaging ten of them does not make
the model better**; it makes the number you quote more reliable. Stratification
matters less here because `Outcome` is not far from balanced; on a rarer class
an unstratified fold can hold too few positives to score at all.

The table shows the holdout landing **0.0102** below the test score and ten
folds **0.0053** above it, so the cheapest design missed by more. On both
ten-fold designs `train F1` sits within half a point of validation, so the
model is not memorising its training rows.

<a class="anchor" id="repeated-kfold">

### <font color='#E8800A'>3.2. Repeated K-Fold Cross-Validation</font>
</a>

Repeated K-Fold runs K-Fold several times, with new random folds each time: five folds and ten repeats fit fifty models.

**How is this different from 50 folds?** On 100 samples, 50 folds validate on 2 rows each, while 5 folds repeated 10 times validate on 20 rows each. Both fit fifty models, but only the second scores enough rows at a time for a fold score to carry information.

**Trade-off**: more computation than one K-Fold, which a large dataset may not need.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RepeatedKFold.html'>sklearn.model_selection.RepeatedKFold(n_splits=5, n_repeats=10, random_state=None)</a>

**Definition:**  
Repeated K-Fold cross validator. Repeats K-Fold n times with different randomization in each repetition.

**Common Parameters:**  
- `n_splits`: Number of folds (must be at least 2)
- `n_repeats`: Number of times K-Fold is repeated
- `random_state`: Seed that makes the folds reproducible
</div>

__`Step 15`__ Create `rkf`, a `RepeatedKFold` with `n_splits=6`, `n_repeats=2` and `random_state=RANDOM_STATE`, so the folds are the same on every run.

In [ ]:
rkf = RepeatedKFold(n_splits=6, n_repeats=2, random_state=RANDOM_STATE)

__`Step 16`__ Score `rkf` with __avg_score__, `record` the result in `frame_log`, and show the per-fold validation F1.

In [ ]:
rkf_result = avg_score(rkf, X_train_val, y_train_val)
record(frame_log, rkf_result, X_test, y_test)
rkf_result['validation']

<a class="anchor" id="loo">

### <font color='#E8800A'>3.3. Leave-One-Out Cross-Validation (LOOCV)</font>
</a>

Leave-One-Out is the extreme case of K-Fold where K equals the number of
samples. Each iteration trains on every row except one and scores that one
held-out row.

It suits very small datasets, and this one is not small. On these 3,200 rows it
costs 3,200 fold fits, each refitting the whole pipeline on 3,199 rows, where
Section 3.1 needed ten.

F1 cannot score one row: a row that is not a win, predicted as not a win, has no
F1 of its own. So `pooled=True` tells `avg_score` to score one F1 over all 3,200
held-out predictions.

__`Step 17`__ **Optional.** Do the same with `LeaveOneOut`, passing `pooled=True` to __avg_score__, and record the result in `frame_log`. It runs for about thirty minutes, and no cell below reads its result, so the notebook runs on without it.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneOut.html'>sklearn.model_selection.LeaveOneOut()</a>

**Definition:**  
Leave-One-Out (LOO) cross-validator. Provides train/test indices to split data where each sample is used once as test set while remaining samples form the training set.

**Common Methods:**  
- `get_n_splits(X)`: Returns the number of splitting iterations (equals number of samples)
</div>

In [ ]:
# LOO is appropriate only for small data. Its folds hold one row each, so
# avg_score pools the held-out predictions into a single F1.
loo = LeaveOneOut()
print('LOO fold fits:', loo.get_n_splits(X_train_val),
      '-- this takes about thirty minutes on my computer.')
loo_result = avg_score(loo, X_train_val, y_train_val, pooled=True)
record(frame_log, loo_result, X_test, y_test)

<a class="anchor" id="stratified">

### <font color='#E8800A'>3.4. Group-Aware and Time-Aware Splitting</font>
</a>

Scikit-learn offers many splitters, all used with similar syntax ([full list](https://scikit-learn.org/stable/api/sklearn.model_selection.html)):

| Class Name | Description | Documentation Link |
|-----------|-------------|-------------------|
| **train_test_split** | Split arrays or matrices into random train and test subsets | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) |
| **KFold** | K-Fold cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) |
| **StratifiedKFold** | Class-wise stratified K-Fold cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html) |
| **RepeatedKFold** | Repeated K-Fold cross validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RepeatedKFold.html) |
| **RepeatedStratifiedKFold** | Repeated class-wise stratified K-Fold cross validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RepeatedStratifiedKFold.html) |
| **LeaveOneOut** | Leave-One-Out cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneOut.html) |
| **LeavePOut** | Leave-P-Out cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeavePOut.html) |
| **GroupKFold** | K-fold iterator variant with non-overlapping groups | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) |
| **StratifiedGroupKFold** | Class-wise stratified K-Fold iterator variant with non-overlapping groups | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html) |
| **LeaveOneGroupOut** | Leave One Group Out cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneGroupOut.html) |
| **LeavePGroupsOut** | Leave P Group(s) Out cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeavePGroupsOut.html) |
| **ShuffleSplit** | Random permutation cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ShuffleSplit.html) |
| **StratifiedShuffleSplit** | Class-wise stratified ShuffleSplit cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedShuffleSplit.html) |
| **GroupShuffleSplit** | Shuffle-Group(s)-Out cross-validation iterator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html) |
| **TimeSeriesSplit** | Time Series cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html) |
| **PredefinedSplit** | Predefined split cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.PredefinedSplit.html) |
| **check_cv** | Input checker utility for building a cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.check_cv.html) |


### Try it: our data has both a group and a time structure

`Competition` places every athlete in a competition level, such as `Local Match`
or `Federation League`, and `Edition` orders them in time: the structures
`GroupKFold` and `TimeSeriesSplit` are built for. The **structure of the data**
and the question asked decide which splitter is correct, not which one scores
better. Athletes from one level share whatever that level does to the outcome,
and a later edition cannot be predicted from its own future.

`Athlete Id` is no group key here: the spine was deduplicated on it, so grouping
by it would be a plain `KFold`.

**The group structure.** `GroupKFold` guarantees that no competition
level supplies rows to both sides of a fold. The cell prints the levels each fold
validates on, the count of levels shared per fold, which should be zero, and the
F1 of predicting a win for every row.

In [ ]:
# The key is already in the frame: no re-reading the file.
groups_train_val = (
    champions.loc[X_train_val.index, "Competition"]
    .astype('string')
    .fillna('<missing competition>')
)
gkf = GroupKFold(n_splits=5)

# Held-out levels, shared levels and the always-win F1, per fold.
held_out, shared_levels, win_floor = [], [], []
for train_idx, val_idx in gkf.split(X_train_val, y_train_val, groups=groups_train_val):
    train_levels = set(groups_train_val.iloc[train_idx])
    val_levels = set(groups_train_val.iloc[val_idx])
    held_out.append(sorted(val_levels))
    shared_levels.append(len(train_levels & val_levels))
    y_fold = y_train_val.iloc[val_idx]
    win_floor.append(round(f1_score(y_fold, np.ones(len(y_fold), dtype=int)), 4))

print("Competition groups:", groups_train_val.nunique())
for fold, levels in enumerate(held_out, start=1):
    print(f"fold {fold} validates on: {', '.join(levels)}")
print("Levels shared between training and validation, per fold:", shared_levels)
print("F1 of predicting a win for every row, per fold:", win_floor, "\n")

group_result = avg_score(gkf, X_train_val, y_train_val, groups=groups_train_val)
record(frame_log, group_result, X_test, y_test)
group_result['validation']

The folds average **0.6684**, against **0.8527** for ten random folds on the
same rows. That is below the floor: predicting a win for every athlete scores between
**0.6873** and **0.8405** on these folds, and the model beats that constant on
two of the five.

A random fold puts one level's athletes on both sides, so the model learns what
that level does to the outcome through its `Competition` column. Validating on
unseen levels removes that, and the drop measures what the knowledge was worth.
The test rows share every level with training, so this row's
`estimate minus truth` compares two different questions. Rows with no recorded
`Competition` form their own group.

**The time structure.** `TimeSeriesSplit` guarantees instead that no
fold's training rows come after its validation rows. The cell sorts the rows by
`Edition` and excludes, with a count, the rows whose `Edition` is unknown, which
have no place in the chronology.

In [ ]:
dates_train_val = champions.loc[X_train_val.index, "Edition"]
known_time = dates_train_val.notna()
# kind='stable' keeps the rows of one edition in their current order.
time_order = dates_train_val[known_time].sort_values(kind='stable').index
X_time = X_train_val.loc[time_order]
y_time = y_train_val.loc[time_order]

print('Rows excluded because Edition is unknown:', int((~known_time).sum()))

tscv = TimeSeriesSplit(n_splits=5)
edition_time = dates_train_val.loc[time_order]
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_time), start=1):
    train_editions = sorted({int(edition) for edition in edition_time.iloc[train_idx]})
    val_editions = sorted({int(edition) for edition in edition_time.iloc[val_idx]})
    print(f"fold {fold}: trains on editions {train_editions}, validates on {val_editions}")
print()

time_result = avg_score(tscv, X_time, y_time)
record(frame_log, time_result, X_test, y_test)
time_result['validation']

Each fold trains on a prefix of the chronology and validates on what comes
next, an order a random `KFold` ignores. The chronological folds average
**0.8373**, under the ten random folds and nowhere near the group split's
collapse.

The small gap reflects this demonstration, not the method. Four editions cannot fill five
folds one season each, so the split cuts *inside* editions and every fold printed
above validates on a season it also trained on. A split that held out a whole
season would ask a harder question, and this notebook does not measure it.

<a class="anchor" id="frame-comparison"></a>

### 3.5. Which Design Estimated Best?

The validation column is what each design claimed before the answer was
available, and the test column is the answer.

In [ ]:
show_log(frame_log)

**The test column barely moves.** Every design handed the same 3,200 rows
reports exactly **0.8474**. The test score comes from refitting the same logistic
regression on those rows, so every such design ships the same model. Changing how you
validate changes what you believe about a model. It does not change the model.
The other two test scores come from other rows: the 3,181 dated rows of
`TimeSeriesSplit` and the 2,400 training rows of the holdout.

Of the designs that ask the test split's question, the holdout missed by the
most, **0.0102**, and the K-Fold designs by **0.0072** or less. That order is not
evidence on its own. The test score is one measurement on 800 rows, and the
ten-fold printouts above spread by **0.0145** to **0.0170**, one standard
deviation, more than any gap in the column. Repeating the split pays off through
that spread instead: ten folds score all 3,200 rows where a holdout scores 800
once, so their average carries less of any one split's luck.

**Leave-One-Out is a disappointment.** When run, the optional cell fits 3,200
models in about thirty minutes to report **0.8509**, a miss of **0.0035**.
Twelve repeated folds took a few seconds and missed by less, so at this sample
size Leave-One-Out is not worth what it costs.

**The group row is a different kind of entry.** It validates on competition
levels the model never trained on, so **0.6684** is the number to quote for a
level the model has never seen, and **0.8474** for athletes from the levels this
data contains.

The `absolute error` column is therefore trustworthy only where the design and
the test split ask the same question: decide first what you are estimating, and
only then which splitter estimates it.

<a class="anchor" id="model-comparison">

## <font color='#E8800A'>4. Comparing Models</font>
</a>

Logistic Regression and the Decision Tree Classifier are scored on one splitter, `RepeatedStratifiedKFold` (the stratification of Section 3.1 with the repetition of Section 3.2), so the comparison is fair.

<a class="anchor" id="decision-tree">

### <font color='#E8800A'>4.1. Decision Tree Classifier</font>
</a>

`avg_score` wraps whatever model it is given in a fresh `PreparedEstimator` in every fold, so scoring the tree takes one call.

[Back to TOC](#toc)

<a class="anchor" id="comparison-results">

### <font color='#E8800A'>4.2. Model Comparison Results</font>
</a>

Both models go into `model_log`, which holds only rows measured on `model_cv`.

In [ ]:
# One seeded splitter and one log for every model from here on.
model_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
model_log = []

untuned_result = avg_score(model_cv, X_train_val, y_train_val)
record(model_log, untuned_result, X_test, y_test)

__`Step 18`__ Score a `DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)` on `model_cv` with __avg_score__, and record it in `model_log`.

In [ ]:
# DO IT
dt_model = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
dt_result = avg_score(model_cv, X_train_val, y_train_val, model=dt_model)
record(model_log, dt_result, X_test, y_test)

On the same ten folds the tree fits its training rows better than the logistic
regression, **0.8643** against **0.8573**, and validates worse, **0.8440**
against **0.8494**: a wider train-validation gap, which is how overfitting
starts.

Its test F1 is higher, **0.8507** against **0.8474**, but by less than either
model's fold-to-fold spread, so neither row outranks the other.

<a class="anchor" id="hyperparameter-tuning">

## <font color='#E8800A'>5. Hyperparameter Tuning</font>
</a>

The same algorithm on the same data produces very different models depending on its configuration, its **hyperparameters**, so model selection must choose them as well as the algorithm.

<a class="anchor" id="what-are-hyperparameters">

### <font color='#E8800A'>5.1. What are Hyperparameters?</font>
</a>

The candidates below vary three of them:

**Logistic Regression:**
- `C`: Regularization strength (smaller values = stronger regularization)

**Decision Tree:**
- `max_depth`: Maximum depth of the tree
- `min_samples_leaf`: Minimum samples required at a leaf

[Back to TOC](#toc)

<a class="anchor" id="grid-search">

### <font color='#E8800A'>5.2. Hyperparameter Search with Holdout Validation</font>
</a>

A **grid search** scores every combination in a predefined grid of hyperparameter values and keeps the best; a **random search** samples combinations at random from a grid or a distribution instead.

__`Step 19`__ Define the candidates every search below chooses between: four logistic regressions and four decision trees

In [ ]:
# Define a grid of hyperparameters to search: four logistic regressions and
# four seeded trees, shared by every search below.
candidates = (
    [LogisticRegression(C=C, max_iter=1000) for C in (0.01, 0.1, 1, 10)]
    + [DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf,
                              random_state=RANDOM_STATE)
       for depth in (5, None) for leaf in (1, 25)]
)

for candidate in candidates:
    print(repr(candidate))
print(f"\n{len(candidates)} candidates")

__`Step 20`__ Perform grid search using holdout validation: fit each candidate, preprocessing included, on **X_train** only and score it on **X_val**; the highest validation F1 wins. The **test set is never touched**.

In [ ]:
# Manual grid search with holdout validation
holdout_rows = []
for candidate in candidates:
    trained = PreparedEstimator(
        classification_preprocessor(classification_recipe),
        clone(candidate),
    ).fit(X_train, y_train)
    params = candidate.get_params()
    guesses = trained.predict(X_val)
    holdout_rows.append({
        'family': type(candidate).__name__,
        'C': params.get('C'),
        'max_depth': str(params.get('max_depth')),
        'min_samples_leaf': params.get('min_samples_leaf'),
        'train F1': f1_score(y_train, trained.predict(X_train)),
        'validation accuracy': accuracy_score(y_val, guesses),
        'validation precision': precision_score(y_val, guesses),
        'validation recall': recall_score(y_val, guesses),
        'validation F1': f1_score(y_val, guesses),
        'candidate': repr(candidate),
    })
holdout_results = pd.DataFrame(holdout_rows)

# idxmax returns the first maximum, the same tie rule as avg_score.
best = holdout_results['validation F1'].idxmax()
best_candidate = candidates[best]

print("Grid Search Results (Holdout)")
print("=" * 50)
print(holdout_results[['train F1', 'validation accuracy', 'validation precision',
                       'validation recall', 'validation F1', 'candidate']]
      .to_string(index=False, float_format='{:.4f}'.format))
print(f"\nBest candidate: {holdout_results.loc[best, 'candidate']}")
print(f"Best validation F1: {holdout_results.loc[best, 'validation F1']:.4f}")

__`Step 21`__ Visualize the grid search results

In [ ]:
# Plot every candidate of both families in the course colours, training F1 in
# blue and validation F1 in orange, so the gap between the two reads off each
# setting.
fig, (ax_logistic, ax_tree) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

logistic_rows = holdout_results[holdout_results['family'] == 'LogisticRegression']
ax_logistic.plot(logistic_rows['C'], logistic_rows['train F1'],
                 color=PLOT_BLUE, linewidth=2, marker='o', markersize=8,
                 label='training')
ax_logistic.plot(logistic_rows['C'], logistic_rows['validation F1'],
                 color=PLOT_ORANGE, linewidth=2, marker='o', markersize=8,
                 label='validation')
ax_logistic.set_xscale('log')
ax_logistic.set_xlabel('C (Regularization Parameter)', fontsize=12)
ax_logistic.set_ylabel('F1', fontsize=12)
ax_logistic.set_title('Logistic Regression', fontsize=13, fontweight='bold')
ax_logistic.legend(fontsize=10)

# A depth of None cannot sit on a numeric axis, so each depth is its own line
# style (solid for 5, dashed for None) and the leaf sizes are evenly spaced
# positions.
tree_rows = holdout_results[holdout_results['family'] == 'DecisionTreeClassifier']
leaf_sizes = sorted(tree_rows['min_samples_leaf'].unique())
positions = list(range(len(leaf_sizes)))
for depth, linestyle in (('5', '-'), ('None', '--')):
    rows = tree_rows[tree_rows['max_depth'] == depth]
    ax_tree.plot(positions, rows['train F1'], color=PLOT_BLUE, linewidth=2,
                 marker='o', markersize=8, linestyle=linestyle,
                 label=f'max_depth={depth}, training')
    ax_tree.plot(positions, rows['validation F1'], color=PLOT_ORANGE, linewidth=2,
                 marker='o', markersize=8, linestyle=linestyle,
                 label=f'max_depth={depth}, validation')
ax_tree.set_xticks(positions, [str(int(leaf)) for leaf in leaf_sizes])
ax_tree.set_xlabel('min_samples_leaf', fontsize=12)
ax_tree.set_title('Decision Tree', fontsize=13, fontweight='bold')
ax_tree.legend(fontsize=10)

fig.suptitle('Holdout search: training and validation F1 of every candidate',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

With no limit on depth or leaf size, the tree scores **1.0000** on training and
**0.8193** on validation: it has memorised its training rows. Requiring 25 rows
per leaf closes most of that gap, **0.8751** against **0.8577**, and wins the
holdout search.

A larger `C` weakens the regularization: the logistic regressions' training F1
climbs and then levels off, while their validation F1 peaks at `C=0.1`,
**0.8464**, and falls after it.

__`Step 22`__ Deploy the winner: retrain it on **train + validation** to use more data, and evaluate it on the **test set** for an unbiased performance estimate.

In [ ]:
# Refit the selected configuration on train+validation rows.
final_model = PreparedEstimator(
    classification_preprocessor(classification_recipe),
    clone(best_candidate),
).fit(X_train_val, y_train_val)

best_score = holdout_results.loc[best, 'validation F1']
test_score = f1_score(y_test, final_model.predict(X_test))

print("Final Model Evaluation")
print(f"Deployed: {holdout_results.loc[best, 'candidate']}")
print(f"Validation F1: {best_score:.4f}")
print(f"Test F1: {test_score:.4f}")
print(f"Difference: {abs(best_score - test_score):.4f}")
print("\nA similar validation and test score is evidence, not a guarantee, of generalization.")

---

## <font color='#E8800A'>5.3. Grid Search with Cross-Validation</font> <a class="anchor" id="grid-search-cv"></a>

`avg_score` runs the Section 5.2 search over folds instead of one validation split: given the list of candidates, it returns the one with the highest mean validation F1, refitted on every training row.

__`Step 23`__ Run the same search on `model_cv` with __avg_score__

In [ ]:
# The same eight candidates, scored on model_cv.
search_result = avg_score(model_cv, X_train_val, y_train_val, model=candidates)
print(f"\nWinner: {search_result['name']}")

__`Step 24`__ Deploy the model __avg_score__ returned, and record it in `model_log`

In [ ]:
# avg_score already refitted the winner; record scores it once on the test rows.
record(model_log, search_result, X_test, y_test,
       validation_is="winner's own score")

Ten folds pick the same tree as the single split, at a validation F1 of
**0.8507** rather than **0.8577**. The best logistic regression, `C=10`, scores
**0.8506**: a fourth-decimal margin against fold-to-fold spreads of **0.0138**
and **0.0172**, so the search does not separate the two families.

On the test rows the tree scores **0.8385**, below the **0.8474** of the untuned
logistic regression. That gap is inside the fold-to-fold spread, so it does not
show that tuning hurt, only that this grid bought nothing measurable. The
winner's own score misses its test F1 by **0.0122**, the largest miss in the log
so far.

---

## <font color='#E8800A'>5.4. Using Pipeline with GridSearchCV</font> <a class="anchor" id="pipeline"></a>

A **Pipeline** makes the recipe and the model one estimator, so `GridSearchCV` (or `RandomizedSearchCV`) refits both inside every training fold.

__`Step 25`__ Run the Section 5.3 search with `GridSearchCV` on a Pipeline

In [ ]:
# Pipeline is scikit-learn's own pairing of the recipe and the model.
# GridSearchCV refits both steps inside every training fold.
pipeline = Pipeline([
    ('prepare', classification_preprocessor(classification_recipe)),
    ('classifier', LogisticRegression(max_iter=1000)),
])

# The eight candidates of Section 5.2, written as two grids. The 'classifier'
# entry swaps the model itself, and the entries after it vary its settings.
param_grid_pipeline = [
    {'classifier': [LogisticRegression(max_iter=1000)],
     'classifier__C': [0.01, 0.1, 1, 10]},
    {'classifier': [DecisionTreeClassifier(random_state=RANDOM_STATE)],
     'classifier__max_depth': [5, None],
     'classifier__min_samples_leaf': [1, 25]},
]

grid_search = GridSearchCV(
    pipeline,
    param_grid_pipeline,
    cv=model_cv,
    scoring='f1',
    return_train_score=True,
    error_score='raise',
    verbose=1,
    n_jobs=4,
)

print("Running GridSearchCV with Pipeline...")
grid_search.fit(X_train_val, y_train_val)

grid_winner = repr(grid_search.best_estimator_.named_steps['classifier'])
test_score_pipeline = f1_score(y_test, grid_search.predict(X_test))

print("\nGridSearchCV Results:")
print(f"Best candidate: {grid_winner}")
print(f"Best CV F1: {grid_search.best_score_:.4f}")
print(f"Its training F1: "
      f"{grid_search.cv_results_['mean_train_score'][grid_search.best_index_]:.4f}")
print(f"Test F1: {test_score_pipeline:.4f}")

# The same folds, candidates and metric as Section 5.3, so the same answer.
assert grid_winner == search_result['name']
assert np.isclose(grid_search.best_score_, np.mean(search_result['validation']))
assert np.isclose(test_score_pipeline,
                  f1_score(y_test, search_result['model'].predict(X_test)))
print("\nSame winner, cross-validated F1 and test F1 as the Section 5.3 search.")

---

## <font color='#E8800A'>5.5. The Two Execution Schemas</font> <a class="anchor" id="execution-schemas"></a>

| Standard search, as in Sections 5.2 to 5.4 | Cached fold search, `CachedSearchCV` |
|---|---|
| Fold 1, candidate A: fit preprocessing, fit model | Fold 1: fit preprocessing once |
| Fold 1, candidate B: fit the same preprocessing again | Cache training and validation matrices |
| Repeat for every candidate and fold | Run candidates A, B, and the rest on those matrices |

Both schemas keep validation rows out of fitting; only the number of
preprocessing fits differs.

[Back to TOC](#toc)

<div class="alert alert-block alert-warning">

**A correct standard search can still repeat work.** Cloning the complete
estimator for every candidate refits the encoder, fills and scaler even when
their settings are identical: redundant work, but no leakage. The cached schema
fits each distinct preprocessing choice once per fold instead.

</div>

__`Step 26`__ Run the Section 5.3 search both ways, time them, and
check that only the cost changed.

In [ ]:
# The Section 5.3 search both ways: the candidates fill the Pipeline's
# 'classifier' step and the PreparedEstimator's 'estimator'. Both run on one
# worker, so the times compare the work and not the threads.
standard_template = GridSearchCV(
    pipeline,
    {"classifier": candidates},
    scoring="f1",
    cv=model_cv,
    error_score="raise",
)
cached_template = CachedSearchCV(
    PreparedEstimator(
        classification_preprocessor(classification_recipe),
        LogisticRegression(max_iter=1000),
    ),
    {"estimator": candidates},
    scoring="f1",
    cv=model_cv,
    n_jobs=1,
)


def timed_fit(search):
    """Return a fresh fitted search and its wall-clock duration in seconds."""
    started = perf_counter()
    fitted = clone(search).fit(X_train_val, y_train_val)
    return fitted, perf_counter() - started


standard_search, standard_seconds = timed_fit(standard_template)
cached_search, cached_seconds = timed_fit(cached_template)

same_scores = np.allclose(
    standard_search.cv_results_["mean_test_score"],
    cached_search.cv_results_["mean_test_score"],
)
standard_fits = model_cv.get_n_splits() * len(candidates)
cached_fits = model_cv.get_n_splits()
speed_up = standard_seconds / cached_seconds

assert same_scores, "the efficient execution changed the candidate scores"
assert speed_up > 1.0, "the cached execution must be faster on this benchmark"
# Both schemas crown the Section 5.3 winner, with the same F1.
assert repr(cached_search.best_estimator_.estimator) == search_result["name"]
assert np.isclose(cached_search.best_score_, np.mean(search_result["validation"]))

# Both score columns side by side, one row per candidate.
comparison = pd.DataFrame({
    "standard": standard_search.cv_results_["mean_test_score"],
    "cached": cached_search.cv_results_["mean_test_score"],
})
comparison["difference"] = comparison["standard"] - comparison["cached"]
comparison["candidate"] = [repr(model) for model in candidates]

print(f"standard: {standard_fits} fold preprocessing fits, {standard_seconds:.1f}s")
print(f"cached:   {cached_fits} fold preprocessing fits, {cached_seconds:.1f}s")
print(f"observed speed-up: {speed_up:.2f}x")
print(f"largest difference between the two score columns:"
      f" {comparison['difference'].abs().max():.2e}\n")
print(comparison.to_string(index=False,
                           formatters={"standard": "{:.6f}".format,
                                       "cached": "{:.6f}".format,
                                       "difference": "{:+.2e}".format}))

Every candidate scores the same in both columns to the last digit printed: the
cache removes repeated work, not evidence. Eighty fold preprocessing fits become
ten, while the eighty fold model fits remain. The seconds depend on the machine;
the removed fits do not.

---

## <font color='#E8800A'>Gold standard: Nested Cross-Validation</font> <a class="anchor" id="nested-cv"></a>

[Back to TOC](#toc)

Every search above reported its winner's own score, which is optimistic: it is
the largest of eight noisy measurements, and the more candidates a search tries,
the further that maximum drifts above the truth.

**Nested cross-validation** scores the winner on rows that took no part in
choosing it. An **outer** loop holds a fold back, the **inner** folds run the
whole search on the rest, and the held-back fold scores the winner. Repeat once
per outer fold and average.

Each outer fold searches different training rows and may return a different
winner. **Nested cross-validation does not select a model. It measures a
selection procedure**: how good is what this search returns on data like this?

The procedure measured here is the cached Section 5.3 search, and the outer loop
is `model_cv`, so the nested row in `model_log` shares the folds of every other
row.

__`Step 27`__ Wrap the Section 5.3 search in an outer loop on
`model_cv`, score each outer fold once, and read the estimate.

In [ ]:
# Outer folds from model_cv; inside each, the cached Section 5.3 search.
nested_rows = []

# ===== OUTER LOOP: one estimate of the whole search per fold =====
for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(
    model_cv.split(X_train_val, y_train_val),
    start=1,
):
    X_outer_train = X_train_val.iloc[outer_train_idx]
    X_outer_test = X_train_val.iloc[outer_test_idx]
    y_outer_train = y_train_val.iloc[outer_train_idx]
    y_outer_test = y_train_val.iloc[outer_test_idx]

    # --- inner search: SELECT, using outer-training rows only ------------
    inner = clone(cached_template).fit(X_outer_train, y_outer_train)

    # --- outer step: the refitted winner is scored ONCE ------------------
    nested_rows.append(
        {
            "outer fold": outer_fold,
            "inner F1": inner.best_score_,
            "outer train F1": f1_score(y_outer_train, inner.predict(X_outer_train)),
            "outer F1": f1_score(y_outer_test, inner.predict(X_outer_test)),
            "chosen": repr(inner.best_estimator_.estimator),
        }
    )

nested_results = pd.DataFrame(nested_rows)
print(nested_results.to_string(index=False, float_format="{:.4f}".format))
print(
    "\nNested estimate:",
    f"{nested_results['outer F1'].mean():.4f}",
    "+/-",
    f"{np.std(nested_results['outer F1']):.4f}",
)

**Ten outer folds chose four different candidates, and none of them is the
model you deploy.** Nine chose a logistic regression, at three values of `C`,
and one chose the depth-5 tree, while the search over all of the training rows
returns the tree with 25 rows per leaf.

The column is a diagnostic, not a tie to break. A sharply peaked grid returns
the same cell from any training rows; a flat one returns whichever cell the
noise favoured. This grid is flat. Fold by fold, the inner score need not sit
above the outer one: the optimism that nesting removes is a property of the
procedure on average.

The deployed model comes from running the same procedure once more on all of
the training rows, which Section 5.3 already did. The estimate is a claim about
the procedure, and the deployed model is one run of it. Never keep the best
outer fold's setting: those rows were reserved for grading, so choosing by them
puts the selection back inside the estimate.

__`Step 28`__ Run the same design in one call, once with each search
from Section 5.5. Both searches are estimators, so `cross_validate` can be the
outer loop. Time the two runs and check that only the cost changed.

In [ ]:
# The same outer folds as the loop above, with each Section 5.5 search as
# the inner loop. Both run on one worker, so the times compare the work and not
# the threads.
def timed_nested(search):
    """Return one nested run's outer scores and its duration in seconds."""
    started = perf_counter()
    scores = cross_validate(search, X_train_val, y_train_val, cv=model_cv,
                            scoring="f1", return_train_score=True)
    return scores, perf_counter() - started


outer_folds = model_cv.get_n_splits()
print(f"standard nested search: {outer_folds * standard_fits} fold preprocessing"
      " fits -- this cell takes about six minutes on my computer.")
standard_outer, standard_nested_seconds = timed_nested(standard_template)
cached_outer, cached_nested_seconds = timed_nested(cached_template)

# Both runs equal the loop above on every outer fold.
for outer in (standard_outer, cached_outer):
    assert np.allclose(outer["test_score"], nested_results["outer F1"])
    assert np.allclose(outer["train_score"], nested_results["outer train F1"])
nested_speed_up = standard_nested_seconds / cached_nested_seconds
assert nested_speed_up > 1.0, "the cached execution must be faster on this benchmark"

print(f"standard: {outer_folds * standard_fits} fold preprocessing fits,"
      f" {standard_nested_seconds:.1f}s")
print(f"cached:   {outer_folds * cached_fits} fold preprocessing fits,"
      f" {cached_nested_seconds:.1f}s")
print(f"observed speed-up: {nested_speed_up:.2f}x\n")
print(pd.DataFrame({
    "outer fold": nested_results["outer fold"],
    "standard outer F1": standard_outer["test_score"],
    "cached outer F1": cached_outer["test_score"],
}).to_string(index=False, float_format="{:.4f}".format))
print(f"\nNested estimate, both ways: {cached_outer['test_score'].mean():.4f}"
      f" +/- {np.std(cached_outer['test_score']):.4f}")

The two runs agree on every outer fold, so the cache changes the cost and
nothing else. It saves more than in Section 5.5: each search's eighty fold
preprocessing fits become ten, and the nested design runs one search per outer
fold, so 800 become 100.

__`Step 29`__ Deploy the Section 5.3 model, and record the nested
estimate beside it in `model_log`.

In [ ]:
# The deployed model is the one the Section 5.3 search returned.
nested_result = {
    "frame": type(model_cv).__name__,
    "name": search_result["name"],
    "rows": len(y_train_val),
    "folds": len(nested_results),
    "train": list(nested_results["outer train F1"]),
    "validation": list(nested_results["outer F1"]),
    "model": search_result["model"],
}
print("outer folds chose:", nested_results["chosen"].value_counts().to_dict(), "\n")
record(model_log, nested_result, X_test, y_test, validation_is="nested estimate")

**One procedure, two numbers.** The ten outer models were fitted only to be
scored: the estimate is their mean outer F1, **0.8513**. So the last two rows of
`model_log` hold the same deployed tree and the same test F1, **0.8385**; their
training and validation F1 differ because the nested row reports the ten outer
winners, **0.8587** and **0.8513**.

The winner's own score, **0.8507**, is the best of eight measurements on the
folds that chose it, so it is optimistic in expectation; the nested estimate
comes from folds that chose nothing. On this run the nested estimate is the
higher of the two, so one run does not show the optimism: the gap is far inside
the fold-to-fold spread of **0.0161**. Both rows miss the test F1 because the
one tree deployed scored below what the procedure delivers on average, on a
single draw of 800 rows.

**Nested cross-validation tests whether a parameter grid is worth searching**, a
question the inner search cannot ask about itself. It also makes grids
comparable: each arm carries its own search inside the measurement, so a wider
grid cannot win by drawing more tickets.

Here it puts the whole search at **0.8513**, against **0.8494** for the untuned
logistic regression on the same folds: a gain of two thousandths against a
spread of **0.0161**. The grid was worth searching once, to establish that it
was not worth searching again.

Holdout, K-Fold and the train/validation/test split all remain useful; nested
cross-validation is the design to reach for when the performance claim itself
is the result.

<div class="alert alert-block alert-success">

**Milestone!** You now command the full assessment toolbox: holdout,
train/validation/test, K-Fold and its stratified/repeated variants,
Leave-One-Out, sklearn's search tools, and nested cross-validation as the
gold standard that keeps selection evidence and estimation evidence apart.

</div>

---

## <font color='#E8800A'>The Complete Model Selection Workflow</font> <a class="anchor" id="workflow"></a>

1. **Choose the validation design** (Sections 2 and 3). The splitter is the
   strategy: a holdout is cheapest and noisiest, K-Fold averages over K splits,
   stratification fixes the class balance in every fold, and repetition redraws
   the folds. Rows that share a group or sit in time order require a group-aware
   or time-aware splitter. Reserve the test rows first, and hold the design fixed
   across every candidate you compare.
2. **Compare candidates under that design** (Sections 4 and 5). A model family
   and a hyperparameter setting are the same kind of choice. Read training and
   validation scores together: a wide gap is overfitting, and two close but low
   scores are underfitting.
3. **Refit the winner on every row that was available for selection**, the
   training and validation rows combined.
4. **Report the estimate your design supports.** One test score is one draw.
   When the performance claim is itself the result, quote the nested estimate
   and its spread.

[Back to TOC](#toc)

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

What this session established:

1. **Selection evidence and estimation evidence are different things.** The
   score that chose a setting is optimistic in expectation, which is why nested
   cross-validation exists. On this grid the optimism was too small to see.
2. **A split is a modelling assumption.** Grouped or time-ordered rows need a
   different splitter, not a different model.
3. **Repeating a split shows what one split was worth.** A single train/test
   division is a sample of size one.
4. **One test set of 800 rows is a coarse judge.** It could not rank the random
   designs, it cannot judge `GroupKFold`, which asks a different question, and
   it cannot rank models the folds call a tie.
5. **Put the whole pipeline inside the search.** Anything fitted outside the
   fold is scored on rows it has already seen.
6. **A search should end in a model.** `avg_score` returns the winner refitted
   on every training row, ready to predict new rows.

---

## <font color='#E8800A'>Optional (Advanced): Scikit-learn Hyperparameter Optimization Tools</font> <a class="anchor" id="optional-tools"></a>

Every search in this notebook tried a fixed list of candidates. scikit-learn offers other search tools, and they avoid leakage the way `GridSearchCV` did in Section 5.4: the estimator they search holds the preprocessing.

### Available Optimization Methods

| Class | Description | Use Case | Documentation |
|-------|-------------|----------|---------------|
| **GridSearchCV** | Exhaustive search over parameter grid | Small parameter spaces, guaranteed to find best in grid | [Link](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) |
| **RandomizedSearchCV** | Random sampling from parameter distributions | Large parameter spaces, faster than grid search | [Link](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) |
| **HalvingGridSearchCV** | Successive halving on parameter grid | Large datasets, eliminates poor candidates early | [Link](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.HalvingGridSearchCV.html) |
| **HalvingRandomSearchCV** | Successive halving with random sampling | Very large parameter spaces | [Link](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.HalvingRandomSearchCV.html) |
| **BayesSearchCV** | Bayesian optimization (requires scikit-optimize) | Expensive model training, smart parameter exploration | [Link](https://scikit-optimize.github.io/stable/modules/generated/skopt.BayesSearchCV.html) |

Beyond scikit-learn, [Optuna](https://optuna.org/) searches well past a grid; nothing in this course requires it.

[Back to TOC](#toc)